# Experiment 1: Pipelined KV-Cache Attention

Compare the cost (cycles and energy) of KV-cache attention with and without pipelining.

Three workloads are evaluated against the TPU v4i architecture:
- **Baseline** (`gpt3_175B_kv_cache.yaml`): full context processed in one pass
- **2-chunk pipeline** (`gpt3_175B_kv_cache_pipeline2.yaml`): context split into 2 chunks, accumulated via vector unit
- **8-chunk pipeline** (`gpt3_175B_kv_cache_pipeline8.yaml`): context split into 8 chunks, accumulated via vector unit

Each workload is mapped automatically with AccelForge's mapper.

In [1]:
from accelforge import Spec, examples
from pathlib import Path

In [2]:
def get_cycles(result):
    return float(result.latency())

def get_energy(result):
    return float(result.energy())

def get_component_energy(result, component):
    energy = result.energy(per_component=True)
    return float(energy.get(component, 0))

def get_component_cyles(result, component):
    latency = result.energy(per_component=True)
    return float(latency.get(component, 0))

## P=2 set up

In [3]:
def get_results(qk_results, sm_results, av_results, acc_results):
    # Total Pipeline Cycles
    t0_cycles = get_cycles(qk_results)
    t1_cycles = max(get_cycles(qk_results), get_cycles(sm_results))
    t2_cycles = max(get_cycles(av_results), get_cycles(sm_results))
    t3_cycles = get_cycles(av_results)
    t4_cycles = get_cycles(acc_results)
    
    print("Total Pipeline Cycles: ", t0_cycles+t1_cycles+t2_cycles+t3_cycles+t4_cycles)
    
    # Total Pipeline Energy
    t0_energy = get_energy(qk_results)
    t1_energy = max(get_energy(qk_results), get_energy(sm_results))
    t2_energy = max(get_energy(av_results), get_energy(sm_results))
    t3_energy = get_energy(av_results)
    t4_energy = get_energy(acc_results)
    
    print("Total Pipeline Energy: ", t0_energy+t1_energy+t2_energy+t3_energy+t4_energy)

## Baseline

In [4]:
qk_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_base = qk_spec_base.map_workload_to_arch()

sm_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_base = sm_spec_base.map_workload_to_arch()

av_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_base = av_spec_base.map_workload_to_arch()

acc_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_base = acc_spec_base.map_workload_to_arch()

Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:01<00:00,  1.61s/it]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 13it [00:00, 121.24it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 120.85it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.13it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 238.42it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 883.01it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1059.97it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 6393.76it/s]


Dirty joining mapping(s) valid & optimal! Returning...


# GLB Capacity

## 25% GLB Capacity

In [5]:
qk_spec_25_cap = Spec.from_yaml(
    "../arches/capacity/25_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_25_cap = qk_spec_25_cap.map_workload_to_arch()

sm_spec_25_cap = Spec.from_yaml(
    "../arches/capacity/25_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_25_cap = sm_spec_25_cap.map_workload_to_arch()

av_spec_25_cap = Spec.from_yaml(
    "../arches/capacity/25_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_25_cap = av_spec_25_cap.map_workload_to_arch()

acc_spec_25_cap = Spec.from_yaml(
    "../arches/capacity/25_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_25_cap = acc_spec_25_cap.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 25.76it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 13it [00:00, 125.91it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 125.69it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.26it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 433.70it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 660.10it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 943.60it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 5629.94it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## 50% GLB Capacity

In [6]:
qk_spec_50_cap = Spec.from_yaml(
    "../arches/capacity/50_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_50_cap = qk_spec_50_cap.map_workload_to_arch()

sm_spec_50_cap = Spec.from_yaml(
    "../arches/capacity/50_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_50_cap = sm_spec_50_cap.map_workload_to_arch()

av_spec_50_cap = Spec.from_yaml(
    "../arches/capacity/50_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_50_cap = av_spec_50_cap.map_workload_to_arch()

acc_spec_50_cap = Spec.from_yaml(
    "../arches/capacity/50_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_50_cap = acc_spec_50_cap.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 28.40it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 14it [00:00, 132.36it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 124.70it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.21it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 359.07it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 778.16it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1071.34it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 7695.97it/s]

Dirty joining mapping(s) valid & optimal! Returning...


## 200% GLB Capacity

In [7]:
qk_spec_200_cap = Spec.from_yaml(
    "../arches/capacity/200_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_200_cap = qk_spec_200_cap.map_workload_to_arch()

sm_spec_200_cap = Spec.from_yaml(
    "../arches/capacity/200_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_200_cap = sm_spec_200_cap.map_workload_to_arch()

av_spec_200_cap = Spec.from_yaml(
    "../arches/capacity/200_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_200_cap = av_spec_200_cap.map_workload_to_arch()

acc_spec_200_cap = Spec.from_yaml(
    "../arches/capacity/200_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_200_cap = acc_spec_200_cap.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 28.61it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 14it [00:00, 137.16it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 126.66it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.29it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 386.11it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 977.01it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1064.27it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 5029.14it/s]

Dirty joining mapping(s) valid & optimal! Returning...


## Comparisons

In [15]:
print('\nbase results:')
get_results(qk_results_base, sm_results_base, av_results_base, acc_results_base)

print('\n25 results:')
get_results(qk_results_25_cap, sm_results_25_cap, av_results_25_cap, acc_results_25_cap)

print('\n50 results:')
get_results(qk_results_50_cap, sm_results_50_cap, av_results_50_cap, acc_results_50_cap)

print('\n200 results:')
get_results(qk_results_200_cap, sm_results_200_cap, av_results_200_cap, acc_results_200_cap)

"""
The reason this happens is because all our tensors fit within
the GLB capacity
"""


base results:
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996

25 results:
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996

50 results:
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996

200 results:
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996


'\nThe reason this happens is because all our tensors fit within\nthe GLB capacity\n'

In [16]:
# Compare max tensor footprint vs GLB sizes
M_CHUNK, H, E = 4096, 128, 128
bytes_per_val = 1  # 8-bit

# Largest tensor in QK stage: K_1[M_CHUNK, H, E]
k1_bytes = M_CHUNK * H * E * bytes_per_val
print(f"K_1 size: {k1_bytes / 1e6:.1f} MB")

glbs = {
    "25%":  1024*1024*128*2,
    "50%":  1024*1024*128*4,
    "100%": 1024*1024*128*8,
    "200%": 1024*1024*128*16,
}
for name, size in glbs.items():
    print(f"GLB {name}: {size/1e6:.0f} MB  — K_1 fits: {k1_bytes < size}")

K_1 size: 67.1 MB
GLB 25%: 268 MB  — K_1 fits: True
GLB 50%: 537 MB  — K_1 fits: True
GLB 100%: 1074 MB  — K_1 fits: True
GLB 200%: 2147 MB  — K_1 fits: True


# GLB Bandwidth

## 25%

In [10]:
qk_spec_25_band = Spec.from_yaml(
    "../arches/bandwidth/25_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_25_band = qk_spec_25_band.map_workload_to_arch()

sm_spec_25_band = Spec.from_yaml(
    "../arches/bandwidth/25_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_25_band = sm_spec_25_band.map_workload_to_arch()

av_spec_25_band = Spec.from_yaml(
    "../arches/bandwidth/25_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_25_band = av_spec_25_band.map_workload_to_arch()

acc_spec_25_band = Spec.from_yaml(
    "../arches/bandwidth/25_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_25_band = acc_spec_25_band.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 27.87it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 15it [00:00, 143.47it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 132.04it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.42it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 260.66it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1212.58it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1310.31it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 7145.32it/s]

Dirty joining mapping(s) valid & optimal! Returning...


## 50%

In [11]:
qk_spec_50_band = Spec.from_yaml(
    "../arches/bandwidth/50_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_50_band = qk_spec_50_band.map_workload_to_arch()

sm_spec_50_band = Spec.from_yaml(
    "../arches/bandwidth/50_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_50_band = sm_spec_50_band.map_workload_to_arch()

av_spec_50_band = Spec.from_yaml(
    "../arches/bandwidth/50_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_50_band = av_spec_50_band.map_workload_to_arch()

acc_spec_50_band = Spec.from_yaml(
    "../arches/bandwidth/50_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_50_band = acc_spec_50_band.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 30.71it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 14it [00:00, 138.78it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 129.36it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.33it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 454.67it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 698.35it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1159.61it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 6297.75it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## 200%

In [12]:
qk_spec_200_band = Spec.from_yaml(
    "../arches/bandwidth/200_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_200_band = qk_spec_200_band.map_workload_to_arch()

sm_spec_200_band = Spec.from_yaml(
    "../arches/bandwidth/200_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_200_band = sm_spec_200_band.map_workload_to_arch()

av_spec_200_band = Spec.from_yaml(
    "../arches/bandwidth/200_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_200_band = av_spec_200_band.map_workload_to_arch()

acc_spec_200_band = Spec.from_yaml(
    "../arches/bandwidth/200_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_200_band = acc_spec_200_band.map_workload_to_arch()

Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 30.16it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 15it [00:00, 143.80it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 134.31it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.48it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 456.60it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1237.26it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1195.30it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 8019.70it/s]

Dirty joining mapping(s) valid & optimal! Returning...


In [13]:
print('\nbase results:')
get_results(qk_results_base, sm_results_base, av_results_base, acc_results_base)

print('\n25 results:')
get_results(qk_results_25_band, sm_results_25_band, av_results_25_band, acc_results_25_band)

print('\n50 results:')
get_results(qk_results_50_band, sm_results_50_band, av_results_50_band, acc_results_50_band)

print('\n200 results:')
get_results(qk_results_200_band, sm_results_200_band, av_results_200_band, acc_results_200_band)



base results:
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996

25 results:
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996

50 results:
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996

200 results:
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996


In [17]:
# Check whether the workload is bandwidth-bound or compute-bound
# If compute time >> memory transfer time, bandwidth doesn't matter

M_CHUNK, H, E = 4096, 128, 128
bytes_per_val = 1  # 8-bit

# Total bytes transferred for QK stage (read Q, K_1; write QK_1)
# Q[1, 1, H, E], K_1[1, M_CHUNK, H, E], QK_1[1, 1, M_CHUNK, H]
q_bytes   = 1 * 1 * H * E * bytes_per_val
k1_bytes  = 1 * M_CHUNK * H * E * bytes_per_val
qk1_bytes = 1 * 1 * M_CHUNK * H * bytes_per_val
total_bw_bytes = q_bytes + k1_bytes + qk1_bytes
print(f"QK stage total data transferred: {total_bw_bytes / 1e6:.1f} MB")

# MXU compute: Q * K_1 = 1 x H x E * M_CHUNK ops → M_CHUNK * H * E MACs
mxu_ops = M_CHUNK * H * E
mxu_throughput = 128 * 128 * 2 * 1.05e9  # 128x128 MXU @ 1.05 GHz (ops/s)
compute_time_s = mxu_ops / mxu_throughput
print(f"QK compute time: {compute_time_s * 1e6:.3f} us")

bandwidths = {
    "25%  (512 GB/s)":  512e9,
    "50%  (1024 GB/s)": 1024e9,
    "100% (2048 GB/s)": 2048e9,
    "200% (4096 GB/s)": 4096e9,
}
print()
for name, bw in bandwidths.items():
    mem_time_s = total_bw_bytes / bw
    print(f"BW {name}: mem transfer = {mem_time_s * 1e6:.3f} us  — "
          f"{'MEMORY-BOUND' if mem_time_s > compute_time_s else 'COMPUTE-BOUND (BW doesnt matter)'}")

""""
Got to figure out how to make this rely on GLB instead of DRAM
The workload is memory-bound at DRAM, not GLB. Changing GLB bandwidth
(512→4096 GB/s) has no effect because the DRAM bottleneck dominates
everything downstream. To actually see a difference, you'd need to sweep
MainMemory bandwidth instead.
"""

QK stage total data transferred: 67.6 MB
QK compute time: 1.950 us

BW 25%  (512 GB/s): mem transfer = 132.128 us  — MEMORY-BOUND
BW 50%  (1024 GB/s): mem transfer = 66.064 us  — MEMORY-BOUND
BW 100% (2048 GB/s): mem transfer = 33.032 us  — MEMORY-BOUND
BW 200% (4096 GB/s): mem transfer = 16.516 us  — MEMORY-BOUND


'"\nGot to figure out how to make this rely on GLB instead of DRAM\nThe workload is memory-bound at DRAM, not GLB. Changing GLB bandwidth\n(512→4096 GB/s) has no effect because the DRAM bottleneck dominates\neverything downstream. To actually see a difference, you\'d need to sweep\nMainMemory bandwidth instead.\n'